In [7]:
# Cell 0 — Project Imports

import torch
from torch import nn

In [8]:
# Cell 1 — U-Net 구성 Block 정의

class DoubleConvolution(nn.Module):
    """Spatial size를 유지하며 두 번의 convolution을 적용"""

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
    ) -> None:
        super().__init__()
        
        self.layers = nn.Sequential(
            nn.Conv2d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                in_channels=out_channels,
                out_channels=out_channels,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )
        
    def forward(
        self,
        input_tensor: torch.Tensor,
    ) -> torch.Tensor:
        return self.layers(input_tensor)
    
    
# EncoderBlock
class DownBlock(nn.Module):
    """Skip feature를 생성하고 spatial size를 절반으로 축소"""

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
    ) -> None:
        super().__init__()

        self.convolutions = DoubleConvolution(
            in_channels=in_channels,
            out_channels=out_channels,
        )
        self.pool = nn.MaxPool2d(
            kernel_size=2,
            stride=2,
        )

    def forward(
        self,
        input_tensor: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        skip_features = self.convolutions(input_tensor)
        pooled_features = self.pool(skip_features)

        return skip_features, pooled_features
    
    
# DecoderBlock
class UpBlock(nn.Module):
    """저해상도 feature를 확대하고 encoder skip feature와 결합"""

    def __init__(
        self,
        in_channels: int,
        skip_channels: int,
        out_channels: int,
    ) -> None:
        super().__init__()

        self.upsample = nn.ConvTranspose2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=2,
            stride=2,
        )

        self.convolutions = DoubleConvolution(
            in_channels=out_channels + skip_channels,
            out_channels=out_channels,
        )

    def forward(
        self,
        input_tensor: torch.Tensor,
        skip_features: torch.Tensor,
    ) -> torch.Tensor:
        upsampled_features = self.upsample(input_tensor)

        combined_features = torch.cat(
            (upsampled_features, skip_features),
            dim=1,
        )

        return self.convolutions(combined_features)


test_input = torch.randn(
    2,
    1,
    64,
    64,
)

down_block = DownBlock(
    in_channels=1,
    out_channels=16,
)

test_skip, test_pooled = down_block(test_input)

print("Input:  ", test_input.shape)
print("Skip:   ", test_skip.shape)
print("Pooled: ", test_pooled.shape)


Input:   torch.Size([2, 1, 64, 64])
Skip:    torch.Size([2, 16, 64, 64])
Pooled:  torch.Size([2, 16, 32, 32])


In [9]:
# Cell 2 — Two-Level 2D U-Net 조립

# 64×64 ──skip₁──────────────────────────────────┐
#    ↓ pool                                      │
# 32×32 ──skip₂─────────────────┐                │
#    ↓ pool                     │                │
# 16×16 bottleneck              │                │
#    ↓ upsample                 │                │
# 32×32 ◄───────────────────────┘                │
#    ↓ upsample                                  │
# 64×64 ◄────────────────────────────────────────┘
#    ↓ 1×1 convolution
# 64×64 class logits

class TinyUNet2D(nn.Module):
    """두 단계의 encoder와 decoder를 사용하는 작은 2D U-Net."""

    def __init__(
        self,
        in_channels: int,
        num_classes: int,
        base_channels: int = 16,
    ) -> None:
        super().__init__()
        
        # [B, 1, 64, 64]
        # -> skip_1   [B, 16, 64, 64]
        # -> pooled_1 [B, 16, 32, 32]
        self.down_block_1 = DownBlock(
            in_channels=in_channels,
            out_channels=base_channels,
        )

        # [B, 16, 32, 32]
        # -> skip_2   [B, 32, 32, 32]
        # -> pooled_2 [B, 32, 16, 16]
        self.down_block_2 = DownBlock(
            in_channels=base_channels,
            out_channels=base_channels * 2,
        )

        # [B, 32, 16, 16]
        # -> [B, 64, 16, 16]
        self.bottleneck = DoubleConvolution(
            in_channels=base_channels * 2,
            out_channels=base_channels * 4,
        )

        # input [B, 64, 16, 16]
        # skip  [B, 32, 32, 32]
        # ->    [B, 32, 32, 32]
        self.up_block_2 = UpBlock(
            in_channels=base_channels * 4,
            skip_channels=base_channels * 2,
            out_channels=base_channels * 2,
        )

        # input [B, 32, 32, 32]
        # skip  [B, 16, 64, 64]
        # ->    [B, 16, 64, 64]
        self.up_block_1 = UpBlock(
            in_channels=base_channels * 2,
            skip_channels=base_channels,
            out_channels=base_channels,
        )

        # [B, 16, 64, 64]
        # -> [B, 3, 64, 64]
        self.segmentation_head = nn.Conv2d(
            in_channels=base_channels,
            out_channels=num_classes,
            kernel_size=1,
        )
        
        
    def forward(
        self,
        input_tensor: torch.Tensor,
    ) -> torch.Tensor:
        skip_1, pooled_1 = self.down_block_1(
            input_tensor,
        )

        skip_2, pooled_2 = self.down_block_2(
            pooled_1,
        )

        bottleneck_features = self.bottleneck(
            pooled_2,
        )

        decoded_2 = self.up_block_2(
            input_tensor=bottleneck_features,
            skip_features=skip_2,
        )

        decoded_1 = self.up_block_1(
            input_tensor=decoded_2,
            skip_features=skip_1,
        )

        segmentation_logits = self.segmentation_head(
            decoded_1,
        )

        return segmentation_logits
    
    

model = TinyUNet2D(
    in_channels=1,
    num_classes=3,
    base_channels=16,
)

# [2, 1, 64, 64]
test_images = torch.randn(
    2,
    1,
    64,
    64,
)

# [2, 3, 64, 64]
test_logits = model(
    test_images,
)

# [2, 64, 64]
test_predictions = test_logits.argmax(
    dim=1,
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("Input:               ", test_images.shape)
print("Logits:              ", test_logits.shape)
print("Prediction:          ", test_predictions.shape)
print("Trainable parameters:", trainable_parameters)

Input:                torch.Size([2, 1, 64, 64])
Logits:               torch.Size([2, 3, 64, 64])
Prediction:           torch.Size([2, 64, 64])
Trainable parameters: 117107


In [10]:
# Cell 3 — Synthetic Segmentation Batch 생성

def create_synthetic_segmentation_batch(
    batch_size: int,
    height: int,
    width: int,
    seed: int,
    device: torch.device,
) -> tuple[torch.Tensor, torch.Tensor]:
    """원과 사각형을 포함하는 작은 segmentation batch를 생성"""
    
    random_generator = torch.Generator(
        device=device,
    )
    random_generator.manual_seed(seed)
    
    # Target contract: [B, H, W], dtype=torch.long
    segmentation_targets = torch.zeros(
        batch_size,
        height,
        width,
        dtype=torch.long,
        device=device,
    )
    
    y_coordinates, x_coordinates = torch.meshgrid(
        torch.arange(height, device=device),
        torch.arange(width, device=device),
        indexing="ij",
    )
    
    for sample_index in range(batch_size):
        # Sample마다 물체 위치를 조금씩 바꾼다.
        circle_center_y = 20 + sample_index * 4
        circle_center_x = 20 + sample_index * 3
        circle_radius = 9

        circle_mask = (
            (y_coordinates - circle_center_y) ** 2
            + (x_coordinates - circle_center_x) ** 2
            <= circle_radius**2
        )

        rectangle_top = 38 - sample_index * 2
        rectangle_left = 40 + sample_index * 2

        rectangle_mask = (
            (y_coordinates >= rectangle_top)
            & (y_coordinates < rectangle_top + 16)
            & (x_coordinates >= rectangle_left)
            & (x_coordinates < rectangle_left + 16)
        )

        # 0: background, 1: circle, 2: rectangle
        segmentation_targets[sample_index][circle_mask] = 1
        segmentation_targets[sample_index][rectangle_mask] = 2
        
        
    # Input contract: [B, C=1, H, W], dtype=torch.float32
    input_images = 0.05 * torch.randn(
        batch_size,
        1,
        height,
        width,
        generator=random_generator,
        device=device,
    )
    
# 각 class에 서로 다른 intensity를 부여한다.
    circle_intensity = (
        segmentation_targets == 1
    ).unsqueeze(dim=1).float() * 0.7

    rectangle_intensity = (
        segmentation_targets == 2
    ).unsqueeze(dim=1).float() * 1.2

    input_images = (
        input_images
        + circle_intensity
        + rectangle_intensity
    )

    return input_images, segmentation_targets


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

training_images, training_targets = (
    create_synthetic_segmentation_batch(
        batch_size=2,
        height=64,
        width=64,
        seed=42,
        device=device,
    )
)

class_counts = torch.bincount(
    training_targets.flatten(),
    minlength=3,
)

print("Device:      ", device)
print("Image shape: ", training_images.shape)
print("Image dtype: ", training_images.dtype)
print("Target shape:", training_targets.shape)
print("Target dtype:", training_targets.dtype)
print("Class counts:", class_counts)


Device:       cuda
Image shape:  torch.Size([2, 1, 64, 64])
Image dtype:  torch.float32
Target shape: torch.Size([2, 64, 64])
Target dtype: torch.int64
Class counts: tensor([7174,  506,  512], device='cuda:0')


In [11]:
# Cell 4 — Tiny Overfit Training

def train_tiny_overfit(
    model: TinyUNet2D,
    input_images: torch.Tensor,
    segmentation_targets: torch.Tensor,
    num_steps: int,
    learning_rate: float,
) -> list[float]:
    """동일한 작은 batch를 반복 학습해 pipeline이 작동하는지 검증"""
    
    loss_function = nn.CrossEntropyLoss()
    
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
    )
    
    loss_history: list[float] = []
    
    model.train()
    
    for step in range(1, num_steps + 1):
        optimizer.zero_grad(
            set_to_none=True,
        )
        
        # [B, 1, H, W] -> [B, K, H, W]
        segmentation_logits = model(
            input_images,
        )
        
        loss = loss_function(
            segmentation_logits,  # [B, K, H, W]
            segmentation_targets, # [B, H, W]
        )
        
        loss.backward()
        
        optimizer.step()
        
        loss_history.append(
            loss.item(),
        )
        
        if step == 1 or step % 50 == 0:
            print(
                f"Step {step:3d} | "
                f"Loss {loss.item():.6f}"
            )

    return loss_history


# Model initialization도 재현 가능하게 고정
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

model = TinyUNet2D(
    in_channels=1,
    num_classes=3,
    base_channels=16,
).to(device)

loss_history = train_tiny_overfit(
    model=model,
    input_images=training_images,
    segmentation_targets=training_targets,
    num_steps=200,
    learning_rate=1e-2,
)

print()
print("Initial loss:", loss_history[0])
print("Final loss:  ", loss_history[-1])

Step   1 | Loss 1.018677
Step  50 | Loss 0.006118
Step 100 | Loss 0.001867
Step 150 | Loss 0.001047
Step 200 | Loss 0.000688

Initial loss: 1.0186773538589478
Final loss:   0.000687967985868454


In [12]:
# Cell 5 — Sample별 Class별 Dice 검증

def compute_per_sample_dice(
    predictions: torch.Tensor,
    targets: torch.Tensor,
    class_index: int,
) -> torch.Tensor:
    """각 sample의 특정 class에 대한 Dice를 계산"""
    
    # [B, H, W] Boolean masks for Class_ID
    prediction_masks = predictions == class_index
    target_masks = targets == class_index
    
    # H와 W에 대해서만 합산하고 batch 차원은 유지
    spatial_dimensions = tuple(
        range(1, predictions.ndim)
    )
    
    # |P ∩ T|
    intersections = (
        prediction_masks
        & target_masks
    ).sum(dim=spatial_dimensions)

    # |P|+|T|
    denominators = (
        prediction_masks.sum(dim=spatial_dimensions)
        + target_masks.sum(dim=spatial_dimensions)
    )

    # Prediction과 target이 모두 비어 있으면 Dice를 1로 정의
    dice_scores = torch.where(
        denominators > 0,
        
        # 2 |P ∩ T| / (|P|+|T|)
        2.0 * intersections / denominators,
        
        torch.ones_like(
            denominators,
            dtype=torch.float32,
        ),
    )

    return dice_scores.float()



model.eval()

# Gradient graph를 만들지 않는 evaluation 전용 context다.
with torch.inference_mode():
    final_logits = model(
        training_images,
    )

    final_predictions = final_logits.argmax(
        dim=1,
    )

class_names: dict[int, str] = {
    0: "background",
    1: "circle",
    2: "rectangle",
}

for class_index, class_name in class_names.items():
    per_sample_scores = compute_per_sample_dice(
        predictions=final_predictions,
        targets=training_targets,
        class_index=class_index,
    )

    print(
        f"{class_name:10s} | "
        f"per sample={per_sample_scores.tolist()} | "
        f"mean={per_sample_scores.mean().item():.6f}"
    )

matching_voxels = (
    final_predictions == training_targets
).sum()

total_voxels = training_targets.numel()

pixel_accuracy = (
    matching_voxels.float()
    / total_voxels
)

print()
print("Prediction shape:", final_predictions.shape)
print("Pixel accuracy:  ", pixel_accuracy.item())

background | per sample=[1.0, 1.0] | mean=1.000000
circle     | per sample=[1.0, 1.0] | mean=1.000000
rectangle  | per sample=[1.0, 1.0] | mean=1.000000

Prediction shape: torch.Size([2, 64, 64])
Pixel accuracy:   1.0
